In [2]:
import os
from llama_index.core.llms import CustomLLM, CompletionResponse, CompletionResponseGen
from typing import Any, List, Optional
from groq import Groq, GroqError

from llama_index.core import ServiceContext, VectorStoreIndex
from llama_index.llms.langchain import LangChainLLM
from llama_index.llms.groq import Groq

import requests
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
import nest_asyncio

from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import VectorStoreIndex, Document

from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever, SummaryIndexRetriever, TransformRetriever

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

from llama_parse import LlamaParse
from llama_index.core import SimpleDirectoryReader

from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings


from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [3]:
GROQ_API_KEY = os.environ["GROQ_API_KEY"] 
COHERE_API_KEY = os.environ["COHERE_API_KEY"] 
LLAMA_CLOUD_API_KEY = os.environ["LLAMA_CLOUD_API_KEY"]

In [4]:
llm = Groq(
    model="llama3-groq-70b-8192-tool-use-preview", api_key=GROQ_API_KEY
)

In [5]:
def load_documents(doc_path: Path):
    file_type = doc_path.suffix
    parser = LlamaParse(
        result_type="text",
    )
    file_extractor = {file_type: parser}
    documents = SimpleDirectoryReader(input_files=[doc_path], file_extractor=file_extractor).load_data()

    return documents


In [6]:
doc_path = Path("../../data/Academic-CV-V1.pdf")
documents = load_documents(doc_path)

Started parsing the file under job_id 63775c88-3823-4f65-a845-efc7436fa4f1


In [ ]:
llm = Groq(
    model="llama3-groq-70b-8192-tool-use-preview",
    api_key=GROQ_API_KEY
)

embed_model = CohereEmbedding(
    api_key=COHERE_API_KEY,
    model_name="embed-english-v3.0",
    input_type="search_query",
)

In [8]:

Settings.llm = llm 
Settings.embed_model = embed_model
# Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
# Settings.num_output = 512
# Settings.context_window = 3900

In [9]:
index = VectorStoreIndex.from_documents(
    documents,
    # service_context=service_context,
    model=embed_model,
)

In [10]:
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=3,
    # vector_store_kwargs={"score_threshold": 0.7},
    # mmr_threshold=0.8
)

In [11]:
response_synthesizer = get_response_synthesizer()

In [12]:
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    # node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.7)],
)

In [13]:
response = query_engine.query("who is the student")

In [14]:
pprint(response.response)

'Hossein Golmohammadi'


In [16]:
# Phoenix can display in real time the traces automatically
# collected from your LlamaIndex application.
# Run all of your LlamaIndex applications as usual and traces
# will be collected and displayed in Phoenix.

# setup Arize Phoenix for logging/observability
import llama_index.core
import os

PHOENIX_API_KEY = "<51cf8166b40108ea610:c85755f>"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
llama_index.core.set_global_handler(
    "arize_phoenix", endpoint="https://llamatrace.com/v1/traces"
)

Attempting to instrument while already instrumented


---

In [16]:
from pathlib import Path
from llama_index.core import (
    Document,
    SimpleDirectoryReader,
)
from llama_parse import LlamaParse
import nest_asyncio

nest_asyncio.apply()

In [35]:
class DocumentRetriever:
    def __init__(self, doc_path: Path, result_type="text", ):
        self.doc_path = doc_path
        self.result_type = result_type

    def load_documents(self) -> list[Document]:
        """
        Load and parse documents using LlamaParse.
        """
        file_type = self.doc_path.suffix
        parser = LlamaParse(result_type="text")
        file_extractor = {file_type: parser}
        documents = SimpleDirectoryReader(input_files=[self.doc_path], file_extractor=file_extractor).load_data()
        return documents

In [36]:
doc_path = Path('~/Documents/App-Fee-Waiver.docx')
fr = DocumentRetriever(doc_path=doc_path)
doc = fr.load_documents()

Error while parsing the file '<bytes/buffer>': [Errno 2] No such file or directory: '~/Documents/App-Fee-Waiver.docx'


In [41]:
doc_path = Path('/Users/artmissg/Documents/App-Fee-Waiver.docx')
# doc_path = Path("/Users/artmissg/Documents/Apply/CV/Final/WPI/Hossein-Golmohammadi-CV.pdf")
document_retriever = DocumentRetriever(doc_path)
documents = document_retriever.load_documents()

Started parsing the file under job_id 4adec350-52ba-494d-a36e-40085af77627


In [42]:
documents

[Document(id_='97c1f1f1-9dae-44ba-b52a-982465d50bee', embedding=None, metadata={'file_path': '/Users/artmissg/Documents/App-Fee-Waiver.docx', 'file_name': 'App-Fee-Waiver.docx', 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'file_size': 204464, 'creation_date': '2024-12-29', 'last_modified_date': '2024-10-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text="My name is Hossein Golmohammadi, and I am very interested in applying to the Computer\nScience graduate program at the University of Pittsburgh. However, I am facing a significant\nchallenge regarding the application fee.\n\nAs an Iranian applicant, I find myself in a difficult position due to the current economic situation\nin my country and the internationa

In [45]:
documents[0].text

"My name is Hossein Golmohammadi, and I am very interested in applying to the Computer\nScience graduate program at the University of Pittsburgh. However, I am facing a significant\nchallenge regarding the application fee.\n\nAs an Iranian applicant, I find myself in a difficult position due to the current economic situation\nin my country and the international financial restrictions we face. The application fee of $85\n(approximately 45 million Rials) represents a substantial financial burden, almost equivalent to a\nmonth's salary in Iran [source: timecamp.com/average-salary/iran]. As a full-time student\nwithout employment, this presents a considerable obstacle to my application process.\nFurthermore, the financial sanctions imposed on Iran severely limit my access to international\npayment methods, making it exceptionally challenging to process the fee payment.\n\nDespite these financial constraints, I have continued to pursue educational opportunities through\nvarious means. I hav